# Gridalytics - Error Analysis

Analyze prediction errors: residuals, time-of-day patterns, seasonal breakdown, worst predictions.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import date

from src.data.db.session import get_session
from src.data.db.models import PredictionLog
from src.evaluation.metrics import classify_delhi_season

## 1. Load Prediction Log

In [ ]:
with get_session() as session:
    logs = session.query(PredictionLog).filter(PredictionLog.actual_peak_mw.isnot(None)).order_by(PredictionLog.target_date).all()
    df = pd.DataFrame([{
        'date': l.target_date,
        'predicted_peak': l.predicted_peak_mw,
        'actual_peak': l.actual_peak_mw,
        'predicted_avg': l.predicted_avg_mw,
        'actual_avg': l.actual_avg_mw,
        'peak_error': l.peak_error_mw,
        'mape': l.mape_pct,
        'mae': l.mae_mw,
        'temperature': l.weather_temp_avg,
        'is_holiday': l.is_holiday,
        'notes': l.notes,
    } for l in logs])

print(f'Prediction log: {len(df)} days with actuals')
df.describe().round(2)

## 2. Predicted vs Actual

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['date'], y=df['actual_peak'], name='Actual Peak', line=dict(color='#3b82f6')))
fig.add_trace(go.Scatter(x=df['date'], y=df['predicted_peak'], name='Predicted Peak', line=dict(color='#f59e0b', dash='dash')))
fig.update_layout(title='Predicted vs Actual Peak Demand', yaxis_title='MW', template='plotly_dark')
fig.show()

## 3. Error Distribution

In [ ]:
fig = px.histogram(df, x='peak_error', nbins=30, title='Peak Error Distribution',
                   labels={'peak_error': 'Error (MW): Predicted - Actual'},
                   marginal='box', color_discrete_sequence=['#3b82f6'])
fig.add_vline(x=0, line_dash='dash', line_color='red')
fig.update_layout(template='plotly_dark')
fig.show()

print(f'Mean error: {df["peak_error"].mean():.1f} MW (negative = underprediction)')
print(f'Std error: {df["peak_error"].std():.1f} MW')
print(f'Mean absolute error: {df["peak_error"].abs().mean():.1f} MW')

## 4. MAPE Over Time

In [ ]:
df['rolling_mape_7d'] = df['mape'].rolling(7, min_periods=1).mean()
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['date'], y=df['mape'], name='Daily MAPE', line=dict(color='gray', width=1)))
fig.add_trace(go.Scatter(x=df['date'], y=df['rolling_mape_7d'], name='7-Day Rolling', line=dict(color='#3b82f6', width=2)))
fig.add_hline(y=5, line_dash='dash', line_color='red', annotation_text='5% threshold')
fig.update_layout(title='MAPE Trend (Drift Detection)', yaxis_title='MAPE %', template='plotly_dark')
fig.show()

## 5. Error vs Temperature

In [ ]:
if df['temperature'].notna().any():
    fig = px.scatter(df, x='temperature', y='peak_error', color='mape',
                     title='Error vs Temperature', labels={'peak_error': 'Error (MW)', 'temperature': 'Avg Temp (C)'},
                     color_continuous_scale='RdYlGn_r')
    fig.add_hline(y=0, line_dash='dash', line_color='white')
    fig.update_layout(template='plotly_dark')
    fig.show()

## 6. Worst Predictions

In [ ]:
worst = df.nlargest(10, 'mape')[['date', 'predicted_peak', 'actual_peak', 'peak_error', 'mape', 'temperature', 'notes']]
print('Top 10 Worst Predictions:')
worst.round(1)